In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# ---------------------------------------------------------
# Configuration: UPDATE THESE PATHS
# ---------------------------------------------------------
train_path = '/content/drive/MyDrive/Parkinsons_Research_Project/datasets/uci_voice_data/train_data.txt'
test_path = '/content/drive/MyDrive/Parkinsons_Research_Project/datasets/uci_voice_data/test_data.txt'

# Define where to save the images
overview_img_path = '/content/drive/MyDrive/Parkinsons_Research_Project/datasets/Table1_Voice_Overview.jpeg'
summary_img_path = '/content/drive/MyDrive/Parkinsons_Research_Project/datasets/Table2_Voice_Features.jpeg'
# ---------------------------------------------------------

# Define the exact 26 features based on your dataset description
feature_names = [
    'Jitter (local)', 'Jitter (local, absolute)', 'Jitter (rap)', 'Jitter (ppq5)', 'Jitter (ddp)',
    'Shimmer (local)', 'Shimmer (local, dB)', 'Shimmer (apq3)', 'Shimmer (apq5)', 'Shimmer (apq11)', 'Shimmer (dda)',
    'AC', 'NTH', 'HTN',
    'Median pitch', 'Mean pitch', 'Standard deviation', 'Minimum pitch', 'Maximum pitch',
    'Number of pulses', 'Number of periods', 'Mean period', 'Standard deviation of period',
    'Fraction of unvoiced frames', 'Number of voice breaks', 'Degree of voice breaks'
]

# Define column structures for Train (29 cols) and Test (28 cols)
train_cols = ['Subject_ID'] + feature_names + ['UPDRS', 'Class']
test_cols = ['Subject_ID'] + feature_names + ['Class']

# Function to render a DataFrame as a high-quality image
def save_df_as_image(df, filename, title=""):
    # Dynamically adjust height based on number of rows (useful for 26 features)
    fig_height = len(df) * 0.35 + 1.5
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.axis('off')

    if title:
        plt.title(title, fontweight='bold', fontsize=16, pad=20)

    table = ax.table(cellText=df.values,
                     colLabels=df.columns,
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])

    table.auto_set_font_size(False)
    table.set_fontsize(11)

    # Style headers and alternating rows
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#dddddd')
        if row == 0:
            cell.set_facecolor('#40466e')
            cell.get_text().set_color('white')
            cell.get_text().set_weight('bold')
        else:
            cell.set_facecolor('#f9f9f9' if row % 2 == 0 else 'white')

    plt.savefig(filename, format='jpeg', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

try:
    # 2. Load the datasets (Assuming comma-separated. If space-separated, change sep=',' to sep='\s+')
    df_train = pd.read_csv(train_path, header=None, names=train_cols, sep=',')
    df_test = pd.read_csv(test_path, header=None, names=test_cols, sep=',')
    print("✅ Train and Test datasets loaded successfully!\n")

    # Combine datasets for overall feature analysis (dropping UPDRS since Test doesn't have it)
    df_combined = pd.concat([df_train.drop(columns=['UPDRS']), df_test], ignore_index=True)

    # 3. Create Dataset Overview Table
    overview_data = {
        'Dataset Partition': ['Training Set', 'Testing Set', 'Total Combined'],
        'Total Recordings': [len(df_train), len(df_test), len(df_combined)],
        'Healthy Controls (Class 0)': [
            len(df_train[df_train['Class'] == 0]),
            len(df_test[df_test['Class'] == 0]),
            len(df_combined[df_combined['Class'] == 0])
        ],
        'Parkinson\'s (Class 1)': [
            len(df_train[df_train['Class'] == 1]),
            len(df_test[df_test['Class'] == 1]),
            len(df_combined[df_combined['Class'] == 1])
        ]
    }
    overview_df = pd.DataFrame(overview_data)

    print("=== Table 1: Dataset Partition Overview ===")
    display(HTML(overview_df.to_html(index=False, classes='table table-striped')))
    save_df_as_image(overview_df, overview_img_path, title="Table 1: Voice Dataset Partition Overview")
    print(f"📸 Saved image to: {overview_img_path}\n")

    # 4. Create Feature Comparison Table (Using the combined dataset)
    print("=== Table 2: Clinical Feature Statistical Summary ===")

    summary_stats = []

    for feature in feature_names:
        healthy_data = df_combined[df_combined['Class'] == 0][feature]
        pd_data = df_combined[df_combined['Class'] == 1][feature]

        healthy_str = f"{healthy_data.mean():.4f} ± {healthy_data.std():.4f}" if len(healthy_data) > 0 else "N/A"
        pd_str = f"{pd_data.mean():.4f} ± {pd_data.std():.4f}" if len(pd_data) > 0 else "N/A"

        summary_stats.append({
            'Acoustic Feature': feature,
            'Healthy Controls (Mean ± SD)': healthy_str,
            'Parkinson\'s Patients (Mean ± SD)': pd_str
        })

    summary_df = pd.DataFrame(summary_stats)

    # Display and Save Feature Summary Table
    display(HTML(summary_df.to_html(index=False, classes='table table-bordered')))
    save_df_as_image(summary_df, summary_img_path, title="Table 2: Acoustic Feature Statistical Summary")
    print(f"📸 Saved image to: {summary_img_path}\n")

except FileNotFoundError as e:
    print(f"❌ Error: Could not find the file. {e}")
except Exception as e:
    print(f"❌ An error occurred: {e}")
    print("If your text files are separated by spaces instead of commas, change `sep=','` to `sep='\\s+'` in the read_csv lines.")

Mounted at /content/drive
✅ Train and Test datasets loaded successfully!

=== Table 1: Dataset Partition Overview ===


Dataset Partition,Total Recordings,Healthy Controls (Class 0),Parkinson's (Class 1)
Training Set,1040,520,520
Testing Set,168,0,168
Total Combined,1208,520,688


📸 Saved image to: /content/drive/MyDrive/Parkinsons_Research_Project/datasets/Table1_Voice_Overview.jpeg

=== Table 2: Clinical Feature Statistical Summary ===


Acoustic Feature,Healthy Controls (Mean ± SD),Parkinson's Patients (Mean ± SD)
Jitter (local),2.5074 ± 1.8312,2.3117 ± 1.7895
"Jitter (local, absolute)",0.0002 ± 0.0001,0.0002 ± 0.0001
Jitter (rap),1.1382 ± 1.0271,1.1067 ± 0.9368
Jitter (ppq5),1.2221 ± 1.2348,1.2061 ± 1.0375
Jitter (ddp),3.4147 ± 3.0814,3.3200 ± 2.8106
Shimmer (local),12.8690 ± 6.1087,11.0619 ± 5.5057
"Shimmer (local, dB)",1.1835 ± 0.4684,1.0276 ± 0.4661
Shimmer (apq3),5.7543 ± 3.4671,4.9207 ± 2.6602
Shimmer (apq5),8.0928 ± 5.8773,6.6950 ± 3.7848
Shimmer (apq11),11.3354 ± 5.5552,10.9257 ± 6.7721


📸 Saved image to: /content/drive/MyDrive/Parkinsons_Research_Project/datasets/Table2_Voice_Features.jpeg

